# Vérification de l'environnement

Ce notebook répond à une seule question : **ce poste peut-il exécuter la chaîne du projet, et
sur quoi exactement ?** Il ne mesure rien et ne produit aucun chiffre destiné au rapport.

Trois verdicts :

- `OK` : la vérification passe.
- `ALERTE` : un manque, qui ne bloque qu'une partie du travail. Le corpus absent en est le cas
  type : il empêche d'entraîner, pas de lire le code ni de lancer les tests. Un dépôt fraîchement
  cloné vit avec plusieurs alertes.
- `ECHEC` : une incohérence, pas un manque. Une version installée hors des bornes déclarées, une
  empreinte de corpus qui ne correspond plus, un tirage aléatoire non reproductible. Un `ECHEC`
  invalide un résultat, ou en produira un faux.

Aucun appel réseau, aucun fichier écrit.

In [89]:
from __future__ import annotations

import math
import os
import platform
import random
import sys
import time
import tomllib
from collections import Counter
from importlib import metadata
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OK = "OK"
ALERTE = "ALERTE"
ECHEC = "ECHEC"

#: Chaque verification enregistree, dans l'ordre ou elle a ete faite.
CHECKS: list[tuple[str, str, str]] = []


def record(label: str, state: str, detail: str = "") -> str:
    """Record one verification and print its verdict.

    Args:
        label: What was verified.
        state: One of ``OK``, ``ALERTE`` or ``ECHEC``.
        detail: The measured value, printed next to the verdict. It states what
            was observed, never why it matters: the reasoning belongs to the
            prose, so that a verdict line stays readable when it passes.

    Returns:
        The recorded state.
    """
    CHECKS.append((label, state, detail))
    print(f"[{state:^6}] {label:<46} {detail}")
    return state


def check(label: str, passed: bool, detail: str = "", *, on_failure: str = ECHEC) -> str:
    """Record a boolean verification.

    Args:
        label: What was verified.
        passed: Outcome of the verification.
        detail: The measured value.
        on_failure: State recorded when the verification fails. Pass ``ALERTE``
            for what blocks part of the work only.

    Returns:
        The recorded state.
    """
    return record(label, OK if passed else on_failure, detail)


print(f"racine du depot : {REPO_ROOT}")

racine du depot : c:\Users\brill\Projets\Scolaire\Train-vs-Pre-train


## 1. Interpréteur et environnement virtuel

La borne de version est lue dans `pyproject.toml`, pas recopiée ici.

L'environnement virtuel a sa propre vérification parce que son absence se manifeste ailleurs, sous
un autre nom. Le `Makefile` appelle l'interpréteur du `.venv` : sans lui, la première cible venue
échoue sur un fichier introuvable. Un notebook lancé depuis un autre environnement importera un
`src` installé ailleurs, ou pas du tout.

In [90]:
from packaging.specifiers import SpecifierSet

PYPROJECT = tomllib.loads((REPO_ROOT / "pyproject.toml").read_text(encoding="utf-8"))
REQUIRES_PYTHON = SpecifierSet(PYPROJECT["project"]["requires-python"])

running = platform.python_version()
check(
    "version de Python dans les bornes declarees",
    REQUIRES_PYTHON.contains(running),
    f"{running}, declare : {REQUIRES_PYTHON}",
)

in_venv = sys.prefix != sys.base_prefix
check("environnement virtuel actif", in_venv, sys.prefix)
check(
    "environnement virtuel du depot",
    in_venv and Path(sys.prefix).resolve() == (REPO_ROOT / ".venv").resolve(),
    f"attendu : {REPO_ROOT / '.venv'}",
    on_failure=ALERTE,
)

print()
print(f"interpreteur      {sys.executable}")
print(f"plateforme        {platform.platform()}")
print(f"machine           {platform.machine()}")

[  OK  ] version de Python dans les bornes declarees    3.12.10, declare : <3.14,>=3.12
[  OK  ] environnement virtuel actif                    c:\Users\brill\Projets\Scolaire\Train-vs-Pre-train\.venv
[  OK  ] environnement virtuel du depot                 attendu : c:\Users\brill\Projets\Scolaire\Train-vs-Pre-train\.venv

interpreteur      c:\Users\brill\Projets\Scolaire\Train-vs-Pre-train\.venv\Scripts\python.exe
plateforme        Windows-11-10.0.26200-SP0
machine           AMD64


## 2. Dépendances déclarées contre dépendances installées

Les groupes de `pyproject.toml` sont comparés à ce que `importlib.metadata` voit installé.

- Un paquet **absent** vaut `ALERTE`. Seul le groupe de base est nécessaire pour entraîner : `dev`
  ne sert qu'à la suite de tests, `eda` qu'au notebook d'analyse exploratoire.
- Un paquet **installé hors des bornes** vaut `ECHEC` : le code tourne contre une version que le
  dépôt ne déclare pas. Le plancher `torch>=2.13` répond à GHSA-rrmf-rvhw-rf47.

In [91]:
from packaging.requirements import Requirement

GROUPS: dict[str, list[str]] = {
    "base": PYPROJECT["project"]["dependencies"],
    **PYPROJECT["project"]["optional-dependencies"],
}


def audit_group(requirements: list[str]) -> tuple[list[str], list[str]]:
    """Compare declared requirements with what is installed.

    Args:
        requirements: Requirement strings, as declared in ``pyproject.toml``.

    Returns:
        A pair ``(missing, conflicting)``. A missing distribution is not
        installed at all; a conflicting one is installed at a version the
        declared specifier refuses.
    """
    missing: list[str] = []
    conflicting: list[str] = []
    for raw in requirements:
        requirement = Requirement(raw)
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            missing.append(requirement.name)
            continue
        if not requirement.specifier.contains(installed, prereleases=True):
            conflicting.append(f"{requirement.name} {installed} hors de {requirement.specifier}")
    return missing, conflicting


for group, requirements in GROUPS.items():
    missing, conflicting = audit_group(requirements)
    state = ECHEC if conflicting else ALERTE if missing else OK
    record(
        f"groupe {group}",
        state,
        f"{len(requirements) - len(missing)}/{len(requirements)} installes",
    )
    for line in conflicting:
        print(f"         conflit : {line}")
    if missing:
        print(f"         absents : {', '.join(missing)}")

[  OK  ] groupe base                                    13/13 installes
[  OK  ] groupe dev                                     2/2 installes
[  OK  ] groupe eda                                     3/3 installes


## 3. Torch, matériel et exécution réelle

`describe_hardware()` est la fonction que chaque run enregistre dans MLflow : le poste est décrit
ici dans les mêmes termes qu'un run tracé.

Le **périphérique retenu** par `resolve_device("auto")`. Sur CPU tout fonctionne, mais les durées
ne sont plus comparables à celles de `reports/` et la précision mixte n'ajoute que du coût de
conversion. D'où l'alerte, pas l'échec.

Le **fait que torch calcule**. Sous Windows, une DLL CUDA manquante ou un runtime Visual C++
absent laisse `import torch` passer et fait échouer la première opération. Un produit matriciel de
512 sur 512 tranche en quelques millisecondes.

In [92]:
import torch

from src.utils.device import describe_hardware, resolve_device

for key, value in describe_hardware().items():
    print(f"{key:<18} {value}")
print()

device = resolve_device("auto")
check("acceleration GPU", device.type == "cuda", f"peripherique retenu : {device}", on_failure=ALERTE)

started = time.perf_counter()
matrix = torch.randn(512, 512, device=device)
total = float((matrix @ matrix.T).sum())
elapsed = time.perf_counter() - started
check(
    "torch execute un produit matriciel",
    math.isfinite(total),
    f"512x512 en {elapsed * 1000:.0f} ms sur {device}",
)

torch_version      2.13.0+cu130
cuda_available     True
cuda_version       13.0
gpu_name           NVIDIA GeForce RTX 5060 Laptop GPU
gpu_count          1
gpu_capability     sm_120

[  OK  ] acceleration GPU                               peripherique retenu : cuda
[  OK  ] torch execute un produit matriciel             512x512 en 91 ms sur cuda


'OK'

## 4. Graine et déterminisme

`set_seed` est appelée une fois par expérience, avant la construction des chargeurs et du modèle.
Sans elle, l'ablation sur la taille du corpus n'est plus rejouable et la comparaison entre les
deux familles de modèles perd son sens.

On ensemence, on tire une valeur de chacun des trois générateurs, on ré-ensemence, on retire. Les
deux tirages doivent être identiques.

In [93]:
import numpy as np

from src.utils.seed import DEFAULT_SEED, set_seed


def draw() -> tuple[float, float, float]:
    """Draw one value from each generator seeded by ``set_seed``.

    Returns:
        One draw from ``random``, from ``numpy`` and from ``torch``.
    """
    return random.random(), float(np.random.rand()), float(torch.randn(1))


state = set_seed(DEFAULT_SEED)
first = draw()
set_seed(DEFAULT_SEED)
second = draw()

check("un meme ensemencement redonne le meme tirage", first == second, f"graine {state.seed}")
check(
    "PYTHONHASHSEED fixe",
    os.environ.get("PYTHONHASHSEED") == str(state.seed),
    os.environ.get("PYTHONHASHSEED", "absent"),
)
check(
    "cuDNN en mode deterministe",
    bool(torch.backends.cudnn.deterministic),
    f"benchmark : {torch.backends.cudnn.benchmark}",
)

[  OK  ] un meme ensemencement redonne le meme tirage   graine 42
[  OK  ] PYTHONHASHSEED fixe                            42
[  OK  ] cuDNN en mode deterministe                     benchmark : False


'OK'

## 5. Corpus figé

Le corpus n'est pas seulement présent ou absent : il est **le bon ou un autre**. Un `train.jsonl`
reconstruit par une exécution ultérieure de la préparation, ou tronqué par une copie interrompue,
est indiscernable de l'original sur un listing de répertoire.

Les empreintes du manifeste tranchent. Elles sont recalculées avec la fonction `checksum` qui les
a écrites, sur les exemples relus par le même chargeur que l'entraînement. Quelques secondes pour
une cinquantaine de mégaoctets.

Un corpus absent vaut alerte : le reste du notebook et la suite de tests tournent sans lui. La
commande qui le construit est rappelée dans le message.

In [94]:
from src.data.dataset import MANIFEST_NAME, load_manifest, load_split
from src.data.split import checksum

PROCESSED = REPO_ROOT / "data" / "processed" / "cnn_dailymail"

if not (PROCESSED / MANIFEST_NAME).is_file():
    check(
        "corpus de travail",
        False,
        f"absent de {PROCESSED}, le construire avec : python -m src.data.build",
        on_failure=ALERTE,
    )
else:
    manifest = load_manifest(PROCESSED)
    print(f"corpus            {manifest.name} (source : {manifest.source_dataset})")
    print(f"version           {manifest.dataset_version[:16]}...")
    print(f"graine du tirage  {manifest.seed}")
    print(f"tokenizer         {manifest.tokenizer}")
    print()

    started = time.perf_counter()
    for split, expected in manifest.splits.items():
        examples = load_split(PROCESSED, split)
        intact = checksum(examples) == manifest.split_checksums[split]
        check(
            f"split {split}",
            len(examples) == expected and intact,
            f"{len(examples)}/{expected} exemples, empreinte {'conforme' if intact else 'DIFFERENTE'}",
        )
    print()
    print(f"empreintes recalculees en {time.perf_counter() - started:.1f} s")

corpus            cnn_dailymail (source : abisee/cnn_dailymail)
version           6d611a231f94db52...
graine du tirage  42
tokenizer         t5-small

[  OK  ] split test                                     1000/1000 exemples, empreinte conforme
[  OK  ] split train                                    30000/30000 exemples, empreinte conforme
[  OK  ] split validation                               1000/1000 exemples, empreinte conforme

empreintes recalculees en 3.5 s


## 6. Configurations d'expériences

`discover_experiments` refuse un fichier invalide, et deux fichiers qui déclarent le même nom :
deux runs homonymes écriraient dans le même répertoire, le second effaçant le premier en silence.
Le chargement fait aussi passer les validateurs de cohérence — un run zéro-shot qui déclarerait
une proportion de corpus, un run entraîné qui n'en déclarerait pas.

In [95]:
from src.experiments.config import discover_experiments

experiments = []
try:
    experiments = discover_experiments(REPO_ROOT / "configs" / "experiments")
except (OSError, ValueError) as error:
    check("configurations d'experiences valides", False, str(error).splitlines()[0])
else:
    check("configurations d'experiences valides", True, f"{len(experiments)} fichiers charges")

print()
for config in experiments:
    proportion = (
        "sans entrainement"
        if config.dataset.percentage is None
        else f"{config.dataset.percentage:>3} % du corpus"
    )
    print(f"  {config.experiment.name:<24} {config.model.type:<12} {proportion}")

[  OK  ] configurations d'experiences valides           9 fichiers charges

  pretrained_ft_10         pretrained    10 % du corpus
  pretrained_ft_100        pretrained   100 % du corpus
  pretrained_ft_50         pretrained    50 % du corpus
  pretrained_zero_shot     pretrained   sans entrainement
  scratch_10               scratch       10 % du corpus
  scratch_100              scratch      100 % du corpus
  scratch_100_layers2      scratch      100 % du corpus
  scratch_100_layers6      scratch      100 % du corpus
  scratch_50               scratch       50 % du corpus


## 7. Résultats enregistrés

Une expérience déclarée sans enregistrement porte le statut `NOT_RUN`, jamais une valeur estimée.
Cette section dit lesquelles des expériences déclarées ont un `run.json` sous `reports/results/`,
et à quel statut.

Un run plafonné porte `PARTIAL` et n'entre dans aucun tableau : `find_record` privilégie
l'enregistrement complet, de sorte qu'une répétition rapide ne masque jamais la mesure qu'elle
préparait.

In [96]:
from src.experiments.record import STATUS_NOT_RUN, STATUS_OK, find_record

RESULTS = REPO_ROOT / "reports" / "results"

complete = 0
for config in experiments:
    found = find_record(RESULTS, config.experiment.name)
    status = STATUS_NOT_RUN if found is None else found.status
    complete += int(status == STATUS_OK)
    print(f"  {config.experiment.name:<24} {status}")

print()
check(
    "chaque experience declaree a un enregistrement complet",
    bool(experiments) and complete == len(experiments),
    f"{complete}/{len(experiments)} au statut {STATUS_OK}",
    on_failure=ALERTE,
)

  pretrained_ft_10         NOT_RUN
  pretrained_ft_100        NOT_RUN
  pretrained_ft_50         NOT_RUN
  pretrained_zero_shot     NOT_RUN
  scratch_10               NOT_RUN
  scratch_100              NOT_RUN
  scratch_100_layers2      NOT_RUN
  scratch_100_layers6      NOT_RUN
  scratch_50               NOT_RUN

[ALERTE] chaque experience declaree a un enregistrement complet 0/9 au statut OK


'ALERTE'

## 8. Magasins locaux

Présence des répertoires que la chaîne lit et écrit, vérifiée sur le système de fichiers. Le
serveur MLflow n'est pas joint : interroger l'URI de tracking est le travail de `make mlflow-ui`,
et un contrôle qui dépend du réseau ne distingue plus un serveur éteint d'un poste mal installé.

Chaque manque vaut alerte. Un poste qui n'a jamais entraîné n'a ni `runs/` ni `mlruns/`.

In [97]:
STORES = {
    "magasin MLflow local": REPO_ROOT / "mlflow.db",
    "runs MLflow": REPO_ROOT / "mlruns",
    "resultats d'experiences": REPO_ROOT / "reports" / "results",
    "figures du rapport": REPO_ROOT / "reports" / "figures",
    "checkpoints d'entrainement": REPO_ROOT / "runs",
}

for label, path in STORES.items():
    check(label, path.exists(), str(path), on_failure=ALERTE)

[  OK  ] magasin MLflow local                           c:\Users\brill\Projets\Scolaire\Train-vs-Pre-train\mlflow.db
[ALERTE] runs MLflow                                    c:\Users\brill\Projets\Scolaire\Train-vs-Pre-train\mlruns
[  OK  ] resultats d'experiences                        c:\Users\brill\Projets\Scolaire\Train-vs-Pre-train\reports\results
[  OK  ] figures du rapport                             c:\Users\brill\Projets\Scolaire\Train-vs-Pre-train\reports\figures
[  OK  ] checkpoints d'entrainement                     c:\Users\brill\Projets\Scolaire\Train-vs-Pre-train\runs


## 9. Provenance du code

`describe_provenance()` lit le commit dans git plutôt que dans une constante entretenue à la main.
Chaque run trace ce commit à côté de ses métriques.

Un arbre de travail modifié est signalé, pas refusé. L'alerte dit qu'un score produit maintenant
ne sera pas rattachable à un état du dépôt qu'un tiers puisse retrouver.

In [98]:
from src.tracking.provenance import UNKNOWN, describe_provenance

provenance = describe_provenance(REPO_ROOT)
for key, value in provenance.to_dict().items():
    print(f"{key:<12} {value}")
print()

check("commit connu", provenance.commit != UNKNOWN, provenance.commit[:12])
check(
    "arbre de travail propre",
    provenance.dirty is False,
    UNKNOWN if provenance.dirty is None else str(provenance.dirty).lower(),
    on_failure=ALERTE,
)

git_commit   a8192d1a41b985af4484f739dc1b1329e385b797
git_branch   feat/switch-cnn-dailymail
git_dirty    true

[  OK  ] commit connu                                   a8192d1a41b9
[ALERTE] arbre de travail propre                        true


'ALERTE'

## 10. Verdict

Le compte, puis la liste de ce qui n'est pas `OK`.

In [99]:
counts = Counter(state for _, state, _ in CHECKS)
print(
    f"{len(CHECKS)} verifications : "
    f"{counts[OK]} {OK}, {counts[ALERTE]} {ALERTE}, {counts[ECHEC]} {ECHEC}"
)
print()

for label, state, detail in CHECKS:
    if state != OK:
        print(f"[{state:^6}] {label:<46} {detail}")

if not counts[ALERTE] and not counts[ECHEC]:
    print("Rien a signaler : la chaine complete peut etre lancee depuis ce poste.")

23 verifications : 20 OK, 3 ALERTE, 0 ECHEC

[ALERTE] chaque experience declaree a un enregistrement complet 0/9 au statut OK
[ALERTE] runs MLflow                                    c:\Users\brill\Projets\Scolaire\Train-vs-Pre-train\mlruns
[ALERTE] arbre de travail propre                        true


## Ce que ce notebook ne vérifie pas

Aucun service n'est joint : ni le serveur MLflow, ni Hugging Face. Un `t5-small` absent du cache
local ne sera constaté qu'au premier téléchargement, et la révision épinglée `df1b051c` n'est
vérifiable que par ce téléchargement.

Ni entraînement ni évaluation. Une mémoire GPU suffisante pour un produit matriciel de 512 sur 512
ne dit rien de ce que demande un `batch_size` donné ; le seul contrôle réel est
`make reproduce MODE=quick`.

Ni la suite de tests. `make test` porte sur le code, ce notebook sur ce qui l'entoure : une chaîne
verte ici avec un `pytest` rouge reste une chaîne rouge.